# AQI Big Data (PM2.5) Pipeline Demo

This notebook runs the end-to-end pipeline for the PM2.5 project.
Run cells top to bottom for a clean demo.

**Flow:** Ingest -> ETL -> EDA -> Features -> Train -> Predict


## 0) Quick Setup Check
Make sure you are using the correct Python kernel (venv).


In [ ]:
import sys
print('Python:', sys.version)
print('Executable:', sys.executable)

## 1) Ingestion (OpenAQ API)
This step fetches new raw JSON data and stores it in `data/raw/`.


In [ ]:
!python src/ingest_openaq.py

In [ ]:
!ls -lt data/raw | head -n 5

## 2) ETL (Raw JSON -> Parquet)
Clean and normalize the raw data, then write Parquet.


In [ ]:
!python src/etl.py

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('demo-etl').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
df_clean = spark.read.parquet('data/processed/pm25_clean')
print('Clean records:', df_clean.count())
df_clean.show(5, truncate=False)
spark.stop()

## 3) EDA (Optional)
Run Spark SQL summaries (daily/weekly trends).


In [ ]:
!python src/eda.py

## 4) Feature Engineering
Create time features and lag_1..lag_24 for time-series learning.


In [ ]:
!python src/features.py

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('demo-features').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
df_feat = spark.read.parquet('data/processed/features')
print('Feature records:', df_feat.count())
print('Columns:', df_feat.columns)
df_feat.show(3, truncate=False)
spark.stop()

## 5) Train Model (Spark MLlib)
Time-based split to avoid leakage.


In [ ]:
!python src/train.py

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('demo-metrics').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
df_metrics = spark.read.parquet('data/processed/model_metrics')
df_metrics.show(truncate=False)
spark.stop()

## 6) Predict
Load model and write predictions.


In [ ]:
!python src/predict.py

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('demo-predict').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
df_pred = spark.read.parquet('data/processed/predictions')
df_pred.show(10, truncate=False)
spark.stop()

## Demo Wrap-up
Key message: The strength of this project is the Big Data pipeline and time-series feature engineering.
